In [1]:
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

# ====== CONFIG ======
INPUT_FILE = "risultati_merged_mixed.csv"
OUTPUT_FILE = "risultati_merged_mixed_enriched.csv"

print("="*60)
print("FEATURE AGGREGATION SCRIPT")
print("="*60)

# ====== LOAD ======
print(f"\nCaricamento {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Dataset caricato: {len(df)} righe, {len(df.columns)} colonne")
print(f"Colonne: {list(df.columns)}")

# ====== FEATURE EXTRACTION ======
def extract_token_features(tokens_str, scores_str):
    """Estrae feature aggregate da tokens e token_scores."""
    features = {}
    
    try:
        # Parse JSON strings
        tokens = json.loads(tokens_str) if isinstance(tokens_str, str) else []
        scores = json.loads(scores_str) if isinstance(scores_str, str) else []
        
        # Se vuoti o None, restituisci valori di default
        if not tokens or not scores:
            return {
                'n_tokens': 0,
                'avg_token_length': 0.0,
                'max_token_length': 0.0,
                'sal_mean': 0.0,
                'sal_std': 0.0,
                'sal_max': 0.0,
                'sal_min': 0.0,
                'sal_median': 0.0,
                'sal_q25': 0.0,
                'sal_q75': 0.0,
                'sal_q90': 0.0,
                'sal_q95': 0.0,
                'sal_sum': 0.0,
                'sal_top3_mean': 0.0,
                'sal_top5_mean': 0.0,
                'sal_top10_mean': 0.0,
                'n_tokens_sal_gt_0_5': 0,
                'n_tokens_sal_gt_1_0': 0,
                'n_tokens_sal_gt_2_0': 0,
                'pct_tokens_sal_gt_0_5': 0.0,
                'pct_tokens_sal_gt_1_0': 0.0,
                'pct_tokens_sal_gt_2_0': 0.0
            }
        
        # Converti scores in numpy array
        scores_arr = np.array(scores, dtype=float)
        
        # Token statistics
        features['n_tokens'] = len(tokens)
        token_lengths = [len(t) for t in tokens]
        features['avg_token_length'] = np.mean(token_lengths) if token_lengths else 0.0
        features['max_token_length'] = np.max(token_lengths) if token_lengths else 0.0
        
        # Saliency statistics
        features['sal_mean'] = float(np.mean(scores_arr))
        features['sal_std'] = float(np.std(scores_arr))
        features['sal_max'] = float(np.max(scores_arr))
        features['sal_min'] = float(np.min(scores_arr))
        features['sal_median'] = float(np.median(scores_arr))
        features['sal_q25'] = float(np.percentile(scores_arr, 25))
        features['sal_q75'] = float(np.percentile(scores_arr, 75))
        features['sal_q90'] = float(np.percentile(scores_arr, 90))
        features['sal_q95'] = float(np.percentile(scores_arr, 95))
        features['sal_sum'] = float(np.sum(scores_arr))
        
        # Top-k saliency means
        sorted_scores = np.sort(scores_arr)[::-1]  # Ordine decrescente
        features['sal_top3_mean'] = float(np.mean(sorted_scores[:3])) if len(sorted_scores) >= 3 else features['sal_max']
        features['sal_top5_mean'] = float(np.mean(sorted_scores[:5])) if len(sorted_scores) >= 5 else features['sal_max']
        features['sal_top10_mean'] = float(np.mean(sorted_scores[:10])) if len(sorted_scores) >= 10 else features['sal_max']
        
        # High saliency token counts
        features['n_tokens_sal_gt_0_5'] = int(np.sum(scores_arr > 0.5))
        features['n_tokens_sal_gt_1_0'] = int(np.sum(scores_arr > 1.0))
        features['n_tokens_sal_gt_2_0'] = int(np.sum(scores_arr > 2.0))
        
        # Percentages
        features['pct_tokens_sal_gt_0_5'] = float(features['n_tokens_sal_gt_0_5'] / len(scores_arr))
        features['pct_tokens_sal_gt_1_0'] = float(features['n_tokens_sal_gt_1_0'] / len(scores_arr))
        features['pct_tokens_sal_gt_2_0'] = float(features['n_tokens_sal_gt_2_0'] / len(scores_arr))
        
    except Exception as e:
        print(f"Errore nel parsing: {e}")
        # Ritorna valori di default in caso di errore
        return {k: 0.0 for k in [
            'n_tokens', 'avg_token_length', 'max_token_length',
            'sal_mean', 'sal_std', 'sal_max', 'sal_min', 'sal_median',
            'sal_q25', 'sal_q75', 'sal_q90', 'sal_q95', 'sal_sum',
            'sal_top3_mean', 'sal_top5_mean', 'sal_top10_mean',
            'n_tokens_sal_gt_0_5', 'n_tokens_sal_gt_1_0', 'n_tokens_sal_gt_2_0',
            'pct_tokens_sal_gt_0_5', 'pct_tokens_sal_gt_1_0', 'pct_tokens_sal_gt_2_0'
        ]}
    
    return features

# ====== APPLY AGGREGATION ======
print("\nEstrazione feature aggregate...")
print("Questa operazione potrebbe richiedere qualche minuto...")

feature_list = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    features = extract_token_features(row['tokens'], row['token_scores'])
    feature_list.append(features)

# Crea DataFrame con le nuove feature
features_df = pd.DataFrame(feature_list)

# ====== MERGE ======
print("\nAggiunta feature al dataset originale...")
df_enriched = pd.concat([df, features_df], axis=1)

print(f"\nNuove colonne aggiunte ({len(features_df.columns)}):")
for col in features_df.columns:
    print(f"  - {col}")

# ====== STATISTICS ======
print("\n" + "="*60)
print("STATISTICHE NUOVE FEATURE")
print("="*60)
print(features_df.describe().T)

# ====== SAVE ======
print(f"\nSalvataggio in {OUTPUT_FILE}...")
df_enriched.to_csv(OUTPUT_FILE, index=False)

print("\n" + "="*60)
print("COMPLETATO!")
print("="*60)
print(f"File salvato: {OUTPUT_FILE}")
print(f"Righe: {len(df_enriched)}")
print(f"Colonne originali: {len(df.columns)}")
print(f"Colonne aggiunte: {len(features_df.columns)}")
print(f"Colonne totali: {len(df_enriched.columns)}")
print("="*60)

# ====== VERIFICA CORRELAZIONI (opzionale) ======
print("\nCorrelazione tra nuove feature e target (logit_td):")
correlations = features_df.corrwith(df['logit_td']).sort_values(ascending=False)
print(correlations)

FEATURE AGGREGATION SCRIPT

Caricamento risultati_merged_mixed.csv...
Dataset caricato: 20828 righe, 9 colonne
Colonne: ['numero_frase', 'label', 'orig_text', 'p_td', 'logit_td', 'tokens', 'token_scores', 'source', 'id']

Estrazione feature aggregate...
Questa operazione potrebbe richiedere qualche minuto...


Processing rows: 100%|██████████| 20828/20828 [00:06<00:00, 3264.97it/s]



Aggiunta feature al dataset originale...

Nuove colonne aggiunte (22):
  - n_tokens
  - avg_token_length
  - max_token_length
  - sal_mean
  - sal_std
  - sal_max
  - sal_min
  - sal_median
  - sal_q25
  - sal_q75
  - sal_q90
  - sal_q95
  - sal_sum
  - sal_top3_mean
  - sal_top5_mean
  - sal_top10_mean
  - n_tokens_sal_gt_0_5
  - n_tokens_sal_gt_1_0
  - n_tokens_sal_gt_2_0
  - pct_tokens_sal_gt_0_5
  - pct_tokens_sal_gt_1_0
  - pct_tokens_sal_gt_2_0

STATISTICHE NUOVE FEATURE
                         count        mean        std           min  \
n_tokens               20828.0  104.637555  73.817254  3.000000e+00   
avg_token_length       20828.0    5.050899   0.801886  1.126984e+00   
max_token_length       20828.0   12.023382   1.929193  4.000000e+00   
sal_mean               20828.0    0.056048   0.176325  8.416971e-05   
sal_std                20828.0    0.059518   0.185617  8.482378e-05   
sal_max                20828.0    0.323891   1.029060  5.653734e-04   
sal_min             